In [1]:
import pandas as pd
import math
import warnings
import time
import numpy as np
from tqdm import tqdm
from fuzzywuzzy import fuzz
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
np.random.seed(0)

In [ ]:
start_time = time.time()   
df = pd.read_csv('./full_joined.csv')
df.sort_values(by='time_submit', inplace=True)
print(df.shape)


#remove invalid jobs with same start and end time
df['exit_code'] = df.apply(lambda x:int(x['exit_code_x'])+int(x['exit_code_y']), axis=1)
temp = pd.DataFrame(df.groupby('id_job')['exit_code'].max())
temp['id_job']=temp.index.to_list()
temp.reset_index(drop=True, inplace=True)
temp.rename(columns={'exit_code':'exit_code_grouped'}, inplace=True)
df = df.merge(temp, on = 'id_job', how='inner')
df[df['exit_code_grouped']!=0][['id_job', 'exit_code_x', 'exit_code_y', 'exit_code_grouped']]
df.drop(columns='exit_code', inplace=True)
print(df[df['exit_code_grouped']==0].shape, df.shape)
df = df[df['exit_code_grouped']==0]
df = df[df['time_start_x']!=df['time_end_x']]


def extract(x):
    if not isinstance(x, str):
        if math.isnan(x):
            return -1
    all = x.split(',')
    for idx, item in enumerate(all):
        if item[0:2]=='2=':
            return float(item[2:])
        else:
            continue
    return -1

def extract_kb(x):
    if not isinstance(x, str):
        if math.isnan(x):
            return -1
    all = x.split(',')
    for idx, item in enumerate(all):
        if item[0:2]=='2=':
            return round(int(item[2:])/(1024*1024), 6)
        else:
            continue
    return 'NA'

df['allocated_memory_mb'] = df['tres_req'].apply(extract)
df['used_memory_mb'] = df['tres_usage_in_max'].apply(extract_kb)
df['wait_time'] = df['time_start_x'] - df['time_submit']

#Wall time is timelimit
df['compute_utilised_time_mins'] = (df['time_end_x'] - df['time_start_x'])/60
df['TimeNotUsedButRequested_hrs'] = (df['timelimit'] - df['compute_utilised_time_mins'])/60
df['%_UtilisedTime'] = (df['compute_utilised_time_mins']/df['timelimit'])*100
df['timelimit_hrs_(allocatedtime)'] = df['timelimit']/60

df_walltime = pd.DataFrame(df.groupby('id_job')['%_UtilisedTime'].max())
df_walltime['id_job'] = df_walltime.index.to_list()
df_walltime.reset_index(drop=True, inplace=True)
df_walltime = df_walltime.merge(df[['id_job', 'constraints', 'partition', 'account']].drop_duplicates(subset=['id_job']), on='id_job', how = 'inner')
df_walltime = df_walltime.merge(df.drop_duplicates(subset='id_job')[['id_job', 'timelimit']])

df['gpu_req'] = df['tres_req'].apply(lambda x:0 if (pd.isna(x) or '1001=' not in x) else int(x.split('1001=')[1]))

mem_allocated = pd.DataFrame(df.groupby('id_job')['allocated_memory_mb'].max())
mem_used = pd.DataFrame(df.groupby('id_job')['used_memory_mb'].max())

mem_used['id_job'] = mem_used.index.to_list()
mem_used.reset_index(drop=True, inplace=True)
mem_allocated['id_job'] = mem_allocated.index.to_list()
mem_allocated.reset_index(drop=True, inplace=True)

memory_overall = mem_allocated.merge(mem_used, on='id_job', how='outer')
memory_overall.head()

memory_overall_pos = memory_overall[memory_overall['used_memory_mb']>=0]
memory_overall_pos = memory_overall_pos[memory_overall_pos['allocated_memory_mb']>=0]
memory_overall_pos['memory_efficiency_%'] = memory_overall_pos['used_memory_mb']*100/memory_overall_pos['allocated_memory_mb']

memory_overall_pos_grps = memory_overall_pos.merge(df[['id_job', 'constraints', 'partition', 'account']].drop_duplicates(subset='id_job'), on='id_job', how = 'inner')

wait_time = pd.DataFrame(df.groupby('id_job')['wait_time'].max())
wait_time['id_job'] = wait_time.index.to_list()
wait_time.reset_index(drop=True, inplace=True)
wait_time = wait_time.merge(df[['id_job', 'constraints', 'partition', 'account']].drop_duplicates(subset=['id_job']), how = 'inner', on='id_job')

df_modelling = df_walltime.merge(memory_overall_pos_grps.drop(columns=['partition', 'account', 'constraints']), on = 'id_job', how='inner')
df_modelling = df_modelling.merge(wait_time.drop(columns=['partition', 'account', 'constraints']), on='id_job', how='inner')
df_modelling = df_modelling.merge(df.drop_duplicates(subset='id_job')[['compute_utilised_time_mins', 'id_job', 'id_user', 'id_assoc', 'id_qos', 'id_group', 'cpus_req', 'gpu_req', 'time_submit', 'job_name']], on = 'id_job', how = 'inner')

print(df_modelling.columns)
df_modelling.rename(columns={'timelimit':'wall_time'}, inplace=True)
df_modelling['wall_time_hrs'] = df_modelling['wall_time']/60
df_modelling['compute_utilised_time_hrs'] = df_modelling['compute_utilised_time_mins']/60
df_modelling['allocated_memory_gb'] = df_modelling['allocated_memory_mb']/1024
df_modelling['used_memory_gb'] = df_modelling['used_memory_mb']/1024
df_modelling = df_modelling[df_modelling['wall_time_hrs']>0]
df_modelling['memory_efficiency_%'] = (df_modelling['used_memory_mb']/df_modelling['allocated_memory_mb'])*100
df_modelling['%_UtilisedTime'] = (df_modelling['compute_utilised_time_mins']/df_modelling['wall_time'])*100
print(df_modelling.shape)
print(df_modelling.shape)
end_time = time.time()
print(f"Time taken to process the data: {round(end_time-start_time, 2)} seconds")

(5787585, 92)
(4435553, 93) (5787585, 93)
Index(['%_UtilisedTime', 'id_job', 'constraints', 'partition', 'account',
       'timelimit', 'allocated_memory_mb', 'used_memory_mb',
       'memory_efficiency_%', 'wait_time', 'compute_utilised_time_mins',
       'id_user', 'id_assoc', 'id_qos', 'id_group', 'cpus_req', 'gpu_req',
       'time_submit', 'job_name'],
      dtype='str')
(2105767, 23)
(2105767, 23)
Time taken to process the data: 90.86 seconds


In [ ]:
#create a new column to identify jobs submitted by the same user at the same time (job array)
start_time = time.time()
df_modelling_copy = df_modelling.copy()
df_modelling_copy.sort_values(by = ['id_user', 'time_submit'], inplace=True)
df_modelling_copy['job_array_idx'] = ''
counter = 1

df_modelling_copy["job_array_idx"] = (df_modelling_copy.groupby(["time_submit", "id_user"], sort=False).cumcount().add(1))

df_modelling = df_modelling_copy
del df_modelling_copy
print(df_modelling.shape, df_modelling.columns)
end_time = time.time()
print(f"Time taken to process the data: {round(end_time-start_time, 2)} seconds")
df_modelling.head()

(2105767, 24) Index(['%_UtilisedTime', 'id_job', 'constraints', 'partition', 'account',
       'wall_time', 'allocated_memory_mb', 'used_memory_mb',
       'memory_efficiency_%', 'wait_time', 'compute_utilised_time_mins',
       'id_user', 'id_assoc', 'id_qos', 'id_group', 'cpus_req', 'gpu_req',
       'time_submit', 'job_name', 'wall_time_hrs', 'compute_utilised_time_hrs',
       'allocated_memory_gb', 'used_memory_gb', 'job_array_idx'],
      dtype='str')
Time taken to process the data: 0.62 seconds


,%_UtilisedTime,id_job,constraints,partition,account,wall_time,allocated_memory_mb,used_memory_mb,memory_efficiency_%,wait_time,compute_utilised_time_mins,id_user,id_assoc,id_qos,id_group,cpus_req,gpu_req,time_submit,job_name,wall_time_hrs,compute_utilised_time_hrs,allocated_memory_gb,used_memory_gb,job_array_idx
120698,7.361111e-01,2001909,NaN,versatile,root,120,3750.0,3.011719,0.080313,1,0.883333,0,2,1,0,1,1,1691999755,bash,2.00,0.014722,3.662109,0.002941,1
152380,1.164153e-09,2043927,NaN,admin,root,4294967295,11250.0,2.984375,0.026528,1,0.050000,0,2,1,0,3,0,1692609857,bash,71582788.25,0.000833,10.986328,0.002914,1
152383,1.940255e-09,2043939,NaN,admin,root,4294967295,3750.0,2.990234,0.079740,1,0.083333,0,2,1,0,1,0,1692610295,bash,71582788.25,0.001389,3.662109,0.002920,1
152384,1.358179e-08,2043941,NaN,admin,root,4294967295,7500.0,2.984375,0.039792,2,0.583333,0,2,1,0,2,0,1692610307,bash,71582788.25,0.009722,7.324219,0.002914,1
358430,1.164153e-09,2469419,NaN,admin,root,4294967295,3750.0,1.257812,0.033542,2,0.050000,0,2,1,0,1,0,1697011958,wrap,71582788.25,0.000833,3.662109,0.001228,1


In [17]:
#curate rolling features
def get_rolling_avg(df):
    
    df.sort_values(by=['time_submit', 'id_job'], inplace=True)
    mem_last = df['used_memory_gb'].rolling(1).mean().shift(1)
    time_last = df['compute_utilised_time_hrs'].rolling(1).mean().shift(1)
    mem_rolling_avg = df['used_memory_gb'].expanding().mean().shift(1)
    time_rolling_avg = df['compute_utilised_time_hrs'].expanding().mean().shift(1)
    mem_3avg = df['used_memory_gb'].rolling(3).mean().shift(1)    
    time_3avg =  df['compute_utilised_time_hrs'].rolling(3).mean().shift(1) 
    mem_7avg = df['used_memory_gb'].rolling(7).mean().shift(1)    
    time_7avg =  df['compute_utilised_time_hrs'].rolling(7).mean().shift(1)
    mem_10avg = df['used_memory_gb'].rolling(10).mean().shift(1)    
    time_10avg =  df['compute_utilised_time_hrs'].rolling(10).mean().shift(1)
    
    mem_last.fillna(0, inplace=True)
    time_last.fillna(0, inplace=True)
    mem_rolling_avg.fillna(0,inplace=True)
    time_rolling_avg.fillna(0, inplace=True)
    mem_3avg.fillna(0, inplace=True)
    time_3avg.fillna(0, inplace=True)
    mem_7avg.fillna(0, inplace=True)
    time_7avg.fillna(0, inplace=True)
    mem_10avg.fillna(0, inplace=True)
    time_10avg.fillna(0, inplace=True)
    
    df['last_memuse_gb'] = mem_last
    df['last_timeuse_hrs'] = time_last
    df['mem_rollingmean'] = mem_rolling_avg
    df['time_rollingmean'] = time_rolling_avg
    df['rollingmean_3mem'] = mem_3avg
    df['rollingmean_3time'] = time_3avg
    df['rollingmean_7mem'] = mem_7avg
    df['rollingmean_7time'] = time_7avg
    df['rollingmean_10mem'] = mem_10avg
    df['rollingmean_10time'] = time_10avg
    
    mem_last_percent = df['memory_efficiency_%'].rolling(1).mean().shift(1)
    time_last_percent = df['%_UtilisedTime'].rolling(1).mean().shift(1)
    mem_rolling_avg_percent = df['memory_efficiency_%'].expanding().mean().shift(1)
    time_rolling_avg_percent = df['%_UtilisedTime'].expanding().mean().shift(1)
    mem_3avg_percent = df['memory_efficiency_%'].rolling(3).mean().shift(1)    
    time_3avg_percent =  df['%_UtilisedTime'].rolling(3).mean().shift(1) 
    mem_7avg_percent = df['memory_efficiency_%'].rolling(7).mean().shift(1)    
    time_7avg_percent =  df['%_UtilisedTime'].rolling(7).mean().shift(1)
    mem_10avg_percent = df['memory_efficiency_%'].rolling(10).mean().shift(1)    
    time_10avg_percent =  df['%_UtilisedTime'].rolling(10).mean().shift(1)
    
    mem_last_percent.fillna(0, inplace=True)
    time_last_percent.fillna(0, inplace=True)
    mem_rolling_avg_percent.fillna(0,inplace=True)
    time_rolling_avg_percent.fillna(0, inplace=True)
    mem_3avg_percent.fillna(0, inplace=True)
    time_3avg_percent.fillna(0, inplace=True)
    mem_7avg_percent.fillna(0, inplace=True)
    time_7avg_percent.fillna(0, inplace=True)
    mem_10avg_percent.fillna(0, inplace=True)
    time_10avg_percent.fillna(0, inplace=True)
    
    df['last_memuse_gb%'] = mem_last_percent
    df['last_timeuse_hrs%'] = time_last_percent
    df['mem_rollingmean%'] = mem_rolling_avg_percent
    df['time_rollingmean%'] = time_rolling_avg_percent
    df['rollingmean_3mem%'] = mem_3avg_percent
    df['rollingmean_3time%'] = time_3avg_percent
    df['rollingmean_7mem%'] = mem_7avg_percent
    df['rollingmean_7time%'] = time_7avg_percent
    df['rollingmean_10mem%'] = mem_10avg_percent
    df['rollingmean_10time%'] = time_10avg_percent
    
    return df
start_time = time.time()
all_users = set(df_modelling['id_user'])
df_modelling_copy = pd.DataFrame()
count = 0
for iduser in tqdm(all_users):
    temp = get_rolling_avg(df_modelling[df_modelling['id_user']==iduser])
    df_modelling_copy = pd.concat([df_modelling_copy, temp])
    
df_modelling = df_modelling_copy
del df_modelling_copy
end_time = time.time()
print(f"Time taken to curate rolling features: {round(end_time-start_time, 2)} seconds")

100%|██████████| 321/321 [00:14<00:00, 22.31it/s]

Time taken to curate rolling features: 14.53 seconds


In [18]:
#curate jobname similarity based feature
def get_txt_features(df_modelling):
    filename_stats_user_global = {'userid': {}}
    score_match = 90
    txt_based_memory, txt_based_time = [], []
    
    for i in tqdm(range(df_modelling.shape[0])):
        
        curr_time, curr_gb = df_modelling['compute_utilised_time_hrs'].iloc[i], df_modelling['used_memory_gb'].iloc[i]
        curr_user = df_modelling['id_user'].iloc[i] 
        job_text = df_modelling['job_name'].iloc[i].lower().strip()
        
        if filename_stats_user_global['userid'].get(curr_user,None)==None:
            filename_stats_user_global['userid'][curr_user] = {job_text:[curr_gb, curr_time]}
            txt_based_memory.append(0)
            txt_based_time.append(0)
        else:
            if filename_stats_user_global['userid'][curr_user].get(job_text,None)!=None:
                
                txt_based_memory.append(filename_stats_user_global['userid'][curr_user].get(job_text)[0])
                txt_based_time.append(filename_stats_user_global['userid'][curr_user].get(job_text)[1])
                
                filename_stats_user_global['userid'][curr_user][job_text][0]+=curr_gb
                filename_stats_user_global['userid'][curr_user][job_text][0] = filename_stats_user_global['userid'][curr_user][job_text][0]/2
                filename_stats_user_global['userid'][curr_user][job_text][1]+=curr_time
                filename_stats_user_global['userid'][curr_user][job_text][1] = filename_stats_user_global['userid'][curr_user][job_text][1]/2
            else:
                # filename_stats_user_global['userid'][curr_user][job_text]=[curr_gb, curr_time]
                #fuzzy search
                avg_mem, avg_time = 0,0
                for key, value in filename_stats_user_global['userid'][curr_user].items():
                    if fuzz.ratio(key, job_text)>score_match:
                        if avg_mem==0:
                            avg_mem = value[0]
                        else:
                            avg_mem+=value[0]
                            avg_mem = avg_mem/2
                        
                        if avg_time==0:
                            avg_time = value[1]
                        else:
                            avg_time+=value[1]
                            avg_time = avg_time/2
                        
                txt_based_memory.append(avg_mem)
                txt_based_time.append(avg_time)
                filename_stats_user_global['userid'][curr_user][job_text] = [curr_gb, curr_time]

    df_modelling['txt_based_mem'] = txt_based_memory
    df_modelling['txt_based_time'] = txt_based_time

    print(df_modelling.shape, len(txt_based_memory), len(txt_based_time))
    return df_modelling                   

start_time = time.time()
df_modelling=get_txt_features(df_modelling)
end_time = time.time()
print(f"Time taken to curate jobname similarity based features: {round(end_time-start_time, 2)} seconds")

100%|██████████| 2105767/2105767 [01:58<00:00, 17737.96it/s]


(2105767, 46) 2105767 2105767
Time taken to curate jobname similarity based features: 119.11 seconds


In [19]:
#remove invalid entries
df_modelling = df_modelling[df_modelling['memory_efficiency_%']<101]
df_modelling = df_modelling[df_modelling['%_UtilisedTime']<101]

In [ ]:
test_time_week = 7*24*60*60 #7 days in seconds
df_modelling.sort_values(by='time_submit', inplace=True)
last_utc = max(df_modelling['time_submit'])
df_train, df_blind_test = df_modelling[df_modelling['time_submit']<=last_utc-test_time_week], df_modelling[df_modelling['time_submit']>last_utc-test_time_week]
df_train.to_csv('./train.csv', index=False)
df_blind_test.to_csv('./test.csv', index=False)